In [ ]:
# --- Setup: make the `ecp` support package available -----------------
# Colab opens a single notebook and installs nothing, so fetch `ecp` from
# the public repo if it isn't importable yet. On Binder/local it is already
# installed, so this cell is a fast no-op there.
try:
    import ecp  # noqa: F401
except ModuleNotFoundError:
    import subprocess, sys
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q",
         "git+https://github.com/ramador09/elementary-computational-physics-binder@main"],
        check=True,
    )


# 6.21 Time-Independent Perturbation Theory and Fine Structure

<!-- This single H1 (one per notebook, "# <number> <Title>") is the page's
     title: it sets the sidebar entry, breadcrumb, browser tab, and search
     result. The branded banner below is generated by the shared `ecp`
     package, so the look of every notebook in the series lives in one place. -->

In [ ]:
from ecp.style import header, use_style

use_style()  # apply the series Matplotlib style
header(
    volume="Volume VI — Quantum Mechanics",
    number="6.21",
    title="Time-Independent Perturbation Theory and Fine Structure",
    blurb="When exact is out of reach. Almost no real Hamiltonian can be solved "
    "exactly, but many are a small correction away from one that can — and perturbation "
    "theory turns that smallness into a systematic expansion. We build the method, then "
    "do something a textbook cannot: check every formula against exact diagonalization, "
    "watching it hold and then fail. The reward is hydrogen's fine structure — the tiny "
    "relativistic splittings that finally break the perfect degeneracy we found in the "
    "Coulomb problem.",
    difficulty="advanced",
    estimate="175–215 min",
)

## Notebook overview

Movement V confronts a fact we have been able to dodge until now: **almost nothing is exactly solvable**.
The harmonic oscillator, the hydrogen atom, the particle in a box — these are the rare, precious
exceptions, and every one of them we have already met. A real atom in a real field, two interacting
electrons, a molecule — none has a closed-form solution. What saves us is that a great many hard problems
are *close* to an easy one: $H=H_0+\lambda V$, with $H_0$ solved and $\lambda V$ a small correction.
**Perturbation theory** turns that closeness into a systematic expansion in powers of $\lambda$, and it
is the workhorse approximation of all of physics.

The formulas are famous and simple. The first-order energy shift is just the **average of the
perturbation** in the unperturbed state, $E_n^{(1)}=\langle n^0|V|n^0\rangle$. The second-order shift is a
**sum over virtual transitions**, $E_n^{(2)}=\sum_{m\ne n}|\langle m^0|V|n^0\rangle|^2/(E_n^0-E_m^0)$,
whose energy denominators are the whole story: when two unperturbed levels crowd together, a term blows
up and the expansion fails. That failure is not a bug — it is the theory telling us that the unperturbed
states were the wrong ones, and that we must first find the **good** states by diagonalizing $V$ within
the degenerate subspace. **Degenerate perturbation theory**, notoriously murky in algebra courses, is
transparent on a computer: it is simply `numpy.linalg.eigh` on the perturbation restricted to the
degenerate block, the "secular equation" made concrete (the eigenspace-projection idea of [§6.5](postulates.ipynb)).

Throughout, we do the one thing a pencil-and-paper derivation cannot: **check every prediction against
exact diagonalization**. For any finite model we build the full $H_0+\lambda V$, diagonalize it with
`numpy.linalg.eigvalsh`, and watch the perturbation series agree beautifully at small $\lambda$, then
peel away as $\lambda$ grows — so the *domain of validity* is something we see rather than take on faith.

Then comes the physics, and it is a promise three notebooks in the making. The perfect
$l$-degeneracy of the hydrogen atom ([§6.17](hydrogen-atom.ipynb)) — every orbital of a shell at one energy — is an artifact of
the *exact* $1/r$ potential and its hidden symmetry. Relativity breaks it. Three small corrections, all
of order $\alpha^2$ (the fine-structure constant squared, $\approx5\times10^{-5}$) — the **relativistic
kinetic** correction, the **spin–orbit** coupling $\mathbf L\cdot\mathbf S$ (posed in [§6.18](spin-magnetic.ipynb), evaluated in
[§6.19](addition-angular-momenta.ipynb)), and the **Darwin** term — split the degenerate levels, and their sum depends only on $n$ and $j$.
The $l$-degeneracy is **lifted**: this is the fine structure of the hydrogen spectrum, the sodium D-line
doublet, the splitting of the Balmer lines.

As in every Volume VI notebook, each exercise opens with a short setup and an enumerated list of parts,
each naming the exact operation to reach for — the perturbation formulas coded explicitly,
`numpy.linalg.eigvalsh` (exact benchmark) and `numpy.linalg.eigh` (degenerate-subspace diagonalization),
the matrix elements $\langle m|V|n\rangle$ via `numpy.vdot`, and the $\mathbf L\cdot\mathbf S$ operator
from [§6.19](addition-angular-momenta.ipynb).

> **Conventions.** $\lambda$ is the dimensionless expansion parameter. For the fine structure we use
> atomic-scale energies in eV with the Rydberg $\text{Ry}=13.606\,$eV and $\alpha=1/137.036$. The
> *method* — perturbation theory, checked against exact — is the backbone of this notebook; fine
> structure is its flagship *application*. We foreground the method and the qualitative result (the
> $l$-degeneracy lifted, levels split by $j$) over grinding the $\langle1/r^3\rangle$ integrals — those
> results are quoted, their algebra summarized. See Sakurai & Napolitano and Griffiths (time-independent and
> degenerate PT, hydrogen fine structure); and Notebooks [§6.5](postulates.ipynb) (eigenspace projection — the good basis),
> [§6.17](hydrogen-atom.ipynb) (the $l$-degeneracy now lifted), [§6.18](spin-magnetic.ipynb) (spin–orbit posed), [§6.19](addition-angular-momenta.ipynb) ($\mathbf L\cdot\mathbf S$
> evaluated), [§6.6](pauli-uncertainty.ipynb) (compatible observables).

## Theory in brief

### The setup and the first-order shift

Write $H=H_0+\lambda V$ with $H_0$ solved (states $|n^0\rangle$, energies $E_n^0$) and expand
$E_n=E_n^0+\lambda E_n^{(1)}+\lambda^2 E_n^{(2)}+\cdots$. The first-order energy is

```{math}
:label: eq-pt-first
E_n^{(1)}=\langle n^0|V|n^0\rangle ,
```

the average of the perturbation — the single most-used result in physics.

### Second order and level repulsion

The state mixes in others, and the second-order energy is a sum over virtual transitions,

```{math}
:label: eq-pt-second
E_n^{(2)}=\sum_{m\ne n}\frac{|\langle m^0|V|n^0\rangle|^2}{E_n^0-E_m^0} ,
```

with the **energy denominators** $E_n^0-E_m^0$ as the crux. For the ground state every denominator is
negative, so $E_0^{(2)}<0$: **levels repel**. When two unperturbed levels are close, a denominator is
tiny and the term explodes — the signal that perturbation theory is breaking down.

### Validity, checked against exact diagonalization

When does the expansion deserve trust? Each term of {eq}`eq-pt-second` compares a matrix element of the
perturbation to an unperturbed level spacing, and the series behaves only while every such ratio stays
small:

```{math}
:label: eq-pt-validity
\text{PT valid when}\quad |\lambda\langle m|V|n\rangle|\ll|E_n^0-E_m^0| ,
```

an asymptotic condition we can *test*: build $H_0+\lambda V$, diagonalize it exactly
(`numpy.linalg.eigvalsh`), and compare — agreement at small $\lambda$, error growing like $\lambda^3$,
breakdown as $\lambda V$ approaches the level spacing.

### Degenerate perturbation theory

When $E_n^0$ is shared by several states, the machinery above stalls: {eq}`eq-pt-second` divides by
level spacings that are now exactly zero. The repair is standard (Griffiths and Sakurai & Napolitano
both develop it in full) and fits on one line:

```{math}
:label: eq-pt-degenerate
E_n^0\ \text{degenerate}\ \Rightarrow\ \text{diagonalize } V\big|_{\text{degenerate subspace}}\ (\texttt{numpy.linalg.eigh}) ,
```

the zero denominators mean the unperturbed states are not unique; the resolution is to diagonalize $V$
*within* the degenerate subspace. Its eigenvalues are the first-order splittings (the degeneracy is
lifted), its eigenvectors the **good** states (the correct zeroth-order basis — the one that diagonalizes
$V$, [§6.5](postulates.ipynb), [§6.6](pauli-uncertainty.ipynb)). The "secular equation" is just a small `eigh`.

### Hydrogen fine structure

Three relativistic corrections, all of order $\alpha^2$, split the Coulomb levels of [§6.17](hydrogen-atom.ipynb): the
**relativistic kinetic** correction (the electron moves at $\sim\alpha c$, so $p^2/2m$ understates the
kinetic energy), the **spin–orbit** coupling $\propto\mathbf L\cdot\mathbf S$ ([§6.18](spin-magnetic.ipynb)/[§6.19](addition-angular-momenta.ipynb)), and the
**Darwin** term (an $l=0$-only smearing). Their sum depends only on $n$ and $j$,

```{math}
:label: eq-fine-structure
E_{n,j}=-\frac{\text{Ry}}{n^2}\left[1+\frac{\alpha^2}{n^2}\left(\frac{n}{j+\tfrac12}-\frac34\right)\right] ,
```

so the perfect $l$-degeneracy of [§6.17](hydrogen-atom.ipynb) is **lifted** (states of the same $n$ now split by $j$), while a
residual $j$-degeneracy remains (lifted further by the Lamb shift, QED — a horizon). The scale is
$\alpha^2\text{Ry}\sim10^{-4}\,$eV, tiny against $13.6\,$eV — exactly why perturbation theory is
justified.

## Setup

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from ecp import draw, validate

ACCENT, INK, SOFT = draw.ACCENT, draw.INK, draw.SOFT
RED, BLUE = "#c1121f", "#1d4e89"

RY = 13.605693  # the Rydberg energy in eV
ALPHA = 1 / 137.035999  # the fine-structure constant


def angular_momentum_matrices(j):
    """The angular-momentum matrices Jx, Jy, Jz for a given j (reused from §6.14/§6.19)."""
    dim = int(round(2 * j + 1))
    m = np.arange(j, -j - 1, -1.0)
    Jz = np.diag(m).astype(complex)
    Jp = np.zeros((dim, dim), dtype=complex)
    Jm = np.zeros((dim, dim), dtype=complex)
    for a in range(dim):
        ma = m[a]
        cp = j * (j + 1) - ma * (ma + 1)
        if cp > 1e-12:
            Jp[a - 1, a] = np.sqrt(cp)
        cm = j * (j + 1) - ma * (ma - 1)
        if cm > 1e-12:
            Jm[a + 1, a] = np.sqrt(cm)
    return (Jp + Jm) / 2, (Jp - Jm) / 2j, Jz


def spin_orbit_operator(l, s=0.5):
    """The spin–orbit operator L·S on the l ⊗ s product space (reused from §6.19)."""
    Lx, Ly, Lz = angular_momentum_matrices(l)
    Sx, Sy, Sz = angular_momentum_matrices(s)
    return np.kron(Lx, Sx) + np.kron(Ly, Sy) + np.kron(Lz, Sz)


def matrix_element(V, bra, ket):
    """The matrix element ⟨bra|V|ket⟩, computed with numpy.vdot so the bra is conjugated."""
    return np.vdot(bra, V @ ket)


def pt_first_order(V, n, states):
    """The first-order energy shift E_n^(1) = ⟨n0|V|n0⟩, the average of the perturbation.

    The columns of ``states`` are the unperturbed eigenstates |n0⟩; pass the identity
    when H0 is already diagonal.
    """
    return matrix_element(V, states[:, n], states[:, n]).real


def pt_second_order(V, n, E0, states):
    """The second-order energy shift E_n^(2) = Σ_{m≠n} |⟨m0|V|n0⟩|^2 / (E_n^0 − E_m^0).

    The sum over virtual transitions. Every denominator E_n^0 − E_m^0 must be nonzero:
    this is the non-degenerate formula, and degenerate levels need degenerate_pt.
    """
    total = 0.0
    for m in range(len(E0)):
        if m != n:
            total += abs(matrix_element(V, states[:, m], states[:, n])) ** 2 / (
                E0[n] - E0[m]
            )
    return total


def degenerate_pt(V, subspace_indices, states=None):
    """Degenerate perturbation theory: diagonalize V within the degenerate subspace.

    Restricts V to the states listed in ``subspace_indices`` (columns of ``states``, or
    the standard basis when ``states`` is None) and diagonalizes the block with
    numpy.linalg.eigh. Returns the first-order splittings (the eigenvalues) and the good
    states (the eigenvectors — the correct zeroth-order basis).
    """
    if states is None:
        block = V[np.ix_(subspace_indices, subspace_indices)]
    else:
        S = states[:, subspace_indices]
        block = S.conj().T @ V @ S
    splittings, good = np.linalg.eigh(block)
    return splittings, good


def fine_structure_energy(n, j):
    """The hydrogen fine-structure energy in eV: E_nj = −(Ry/n^2)·[1 + (α^2/n^2)(n/(j+1/2) − 3/4)].

    Depends only on n and j — the l-degeneracy is lifted, and a residual j-degeneracy
    remains (broken only by the Lamb shift).
    """
    return -(RY / n**2) * (1 + (ALPHA**2 / n**2) * (n / (j + 0.5) - 0.75))

## Exercise 1 — First-order perturbation theory, checked against exact

The leading correction to an energy level is the average of the perturbation in the unperturbed state —
the single most-used formula in physics {eq}`eq-pt-first`. We verify it the honest way, against an exact
answer.

1. Build a diagonal $H_0$ (the unperturbed energies) and a symmetric random perturbation $V$.
2. Compute the first-order shift $E_0^{(1)}=\langle 0|V|0\rangle$ for the ground state, using the
   `pt_first_order` helper (a `numpy.vdot` matrix element).
3. Diagonalize $H_0+\lambda V$ exactly with `numpy.linalg.eigvalsh` at a small coupling $\lambda$.
4. Confirm the prediction $E_0^0+\lambda\langle 0|V|0\rangle$ matches the exact ground-state energy.

In [ ]:
# (solution hidden on the public site)


### Validation 1

In [ ]:
validate.close(
    E_pt1,
    E_exact[0],
    "the first-order shift ⟨n|V|n⟩ matches exact diagonalization at small λ",
    atol=1e-2,
)

## Exercise 2 — Second-order perturbation theory and level repulsion

We refine the energy estimate to second order and watch the characteristic "repulsion" of neighbouring
levels emerge {eq}`eq-pt-second`.

1. Compute the second-order shift $E_n^{(2)}=\sum_{m\ne n}|\langle m|V|n\rangle|^2/(E_n^0-E_m^0)$ with
   the `pt_second_order` helper (matrix elements via `numpy.vdot`).
2. Add it to the first-order result and compare both orders against exact diagonalization
   (`numpy.linalg.eigvalsh`) across a range of couplings $\lambda$.
3. Confirm the ground-state second-order shift is negative — every denominator $E_0^0-E_m^0$ is — and
   interpret it as level repulsion.

In [ ]:
# (solution hidden on the public site)


### Validation 2

In [ ]:
validate.check(
    abs(E_pt2 - E_exact0) < abs((E0[0] + lam * E1_shift) - E_exact0) and E2_shift < 0,
    "second-order PT improves the agreement over first order, and the ground state is pushed down (level repulsion)",
)

In [ ]:
# (solution hidden on the public site)


## Exercise 3 — Where perturbation theory breaks

Perturbation theory carries its own failure signal — the energy denominators. Here we close the gap
between two levels and watch the second-order term blow up {eq}`eq-pt-validity`, {eq}`eq-pt-second`.

1. Build two nearly-degenerate levels, $E_0^0=0$ and $E_1^0=\delta$ with $\delta$ small, coupled by an
   off-diagonal $V_{01}$.
2. Compute the second-order shift $|V_{01}|^2/(E_0^0-E_1^0)$ and note it grows as $1/\delta$.
3. Compare against the exact ground state (`numpy.linalg.eigvalsh`) as $\delta$ shrinks, and show the
   overshoot growing.
4. Conclude that non-degenerate perturbation theory is invalid when levels approach — the cue for the
   degenerate treatment of Exercise 4.

In [ ]:
# (solution hidden on the public site)


### Validation 3

In [ ]:
validate.check(
    breakdown_ok,
    "the second-order shift overshoots the exact result for nearly-degenerate levels — PT fails when unperturbed levels are close (small denominators)",
)

## Exercise 4 — Degenerate perturbation theory as subspace diagonalization

When the unperturbed level is degenerate the denominators vanish identically, and the fix is the most
notorious piece of algebra in a quantum course — which on a computer is three lines: the "secular
equation" is an `eigh` on a block, and the good basis is the one that diagonalizes $V$ ([§6.5](postulates.ipynb))
{eq}`eq-pt-degenerate`.

1. Build an $H_0$ with a triply-degenerate level and a random Hermitian perturbation $W$.
2. Restrict $W$ to the degenerate subspace and diagonalize the block with `numpy.linalg.eigh` (the
   `degenerate_pt` helper): the eigenvalues are the first-order splittings, the eigenvectors the good
   states.
3. Confirm the split levels $E^0+\lambda\times(\text{splittings})$ match exact diagonalization of the
   full $H$ (`numpy.linalg.eigvalsh`).

In [ ]:
# (solution hidden on the public site)


### Validation 4

In [ ]:
validate.close(
    pt_levels,
    exact_levels,
    "degenerate PT is diagonalization of the perturbation within the degenerate subspace — matching exact to 1e-3",
    atol=1e-3,
)

In [ ]:
# (solution hidden on the public site)


## Exercise 5 — Hydrogen fine structure: the spin–orbit splitting

The spin–orbit coupling $H_{SO}\propto\mathbf L\cdot\mathbf S$ is the most famous ingredient of the fine
structure, and it is a degenerate-perturbation problem whose good basis we already own: the coupled
$|j,m\rangle$ basis of [§6.19](addition-angular-momenta.ipynb), in which $\mathbf L\cdot\mathbf S$ is diagonal {eq}`eq-fine-structure`.

1. Recall that in the coupled basis $\mathbf L\cdot\mathbf S$ has eigenvalue
   $\tfrac12[j(j+1)-l(l+1)-s(s+1)]$.
2. Build $\mathbf L\cdot\mathbf S$ for $l=1$, $s=\tfrac12$ with the `spin_orbit_operator` helper ([§6.19](addition-angular-momenta.ipynb))
   and read off its distinct eigenvalues with `numpy.linalg.eigvalsh`.
3. Show $j=\tfrac32$ is raised and $j=\tfrac12$ lowered, and match the operator's eigenvalues to the
   formula.
4. Identify the pattern as the sodium-D-type doublet.

In [ ]:
# (solution hidden on the public site)


### Validation 5

In [ ]:
validate.close(
    np.array(sorted(distinct)),
    np.array(sorted(ls_grid)),
    "the spin–orbit coupling splits levels by j (j=l+½ raised, j=l−½ lowered) with eigenvalue ½[j(j+1)−l(l+1)−¾]",
    rtol=1e-9,
)

## Exercise 6 — The full fine-structure formula and the lifted degeneracy

Three relativistic corrections, all of order $\alpha^2$, combine into one closed formula — and the
punchline is which quantum numbers survive in it. The perfect Coulomb degeneracy of [§6.17](hydrogen-atom.ipynb) is about to be
lifted by relativity {eq}`eq-fine-structure`.

1. State the three $\alpha^2$ corrections (relativistic kinetic, spin–orbit, Darwin); their sum is
   $E_{n,j}=-(\text{Ry}/n^2)[1+(\alpha^2/n^2)(n/(j+\tfrac12)-\tfrac34)]$, coded in the
   `fine_structure_energy` helper.
2. Evaluate the $n=2$ states: $2S_{1/2}$, $2P_{1/2}$, $2P_{3/2}$.
3. Show $2S_{1/2}$ and $2P_{1/2}$ (same $j=\tfrac12$, different $l$) *coincide* while $2P_{3/2}$ splits
   off — the energy depends on $j$, not $l$: the $l$-degeneracy is lifted.
4. Compute the $2P_{3/2}-2P_{1/2}$ splitting ($\sim\alpha^2\text{Ry}\sim0.05\,$meV) and note that its
   smallness is precisely what justifies perturbation theory.

In [ ]:
# (solution hidden on the public site)


### Validation 6

In [ ]:
validate.close(
    split_meV,
    0.0453,
    "the fine-structure energy depends only on n and j; the l-degeneracy of §6.17 is lifted (2P₃/₂−2P₁/₂ ≈ 0.045 meV)",
    atol=5e-3,
)

In [ ]:
# (solution hidden on the public site)


## Exercise 7 — Perturbing the harmonic oscillator *(student)*

The general method, turned on a familiar system: the anharmonic oscillator ([§6.12](harmonic-oscillator.ipynb)), whose $\lambda x^4$
correction admits no closed-form spectrum {eq}`eq-pt-first`, {eq}`eq-pt-validity`.

1. Take the oscillator as $H_0$ in the number basis ($E_n=n+\tfrac12$ with $\hbar=\omega=1$).
2. Build $x=(a+a^\dagger)/\sqrt2$ from the ladder operators and form the perturbation $x^4$ with
   `numpy.linalg.matrix_power`.
3. Compute the first-order ground-state shift $\lambda\langle0|x^4|0\rangle$.
4. Compare against exact diagonalization of $H_0+\lambda x^4$ (`numpy.linalg.eigvalsh`) across $\lambda$
   and identify where perturbation theory breaks down.

In [ ]:
# (solution hidden on the public site)


### Validation 7

In [ ]:
validate.check(
    student_ok,
    "perturbation theory matches exact at small λ and diverges as λ grows for the λx⁴ anharmonic oscillator — its accuracy and breakdown, mapped",
)

## Exercise 8 — Approximation with a conscience

Almost nothing in physics is exactly solvable, and perturbation theory is how we make progress anyway —
by expanding around what we *can* solve and keeping the small terms. The formulas are simple: the average
of the perturbation, then a sum over virtual transitions. But their honesty lives in the **denominators**:
when two levels crowd together the expansion fails, and we must first find the good states by
diagonalizing $V$ within the degenerate subspace — degenerate perturbation theory, which on a computer is
just a small `eigh`.

**This exercise (synthesis).** No new computation: the method, and its honesty, are the result. We did
what no derivation can — watched the approximation agree with the exact answer and then part from it — and
then turned it on the real prize: the **fine structure** that splits hydrogen's degenerate levels by a few
parts in a hundred thousand, lifting the $l$-degeneracy the Coulomb symmetry had protected ([§6.17](hydrogen-atom.ipynb)), through
the spin–orbit coupling we posed in [§6.18](spin-magnetic.ipynb) and evaluated in [§6.19](addition-angular-momenta.ipynb). **Movement V has begun.** The next
notebook ([§6.22](variational-method.ipynb)) turns to a different and often more powerful approximation — the **variational method**,
which bounds the ground-state energy of systems too far from any solvable one for perturbation theory to
reach, like the helium atom — and then the semiclassical WKB limit ([§6.23](wkb-semiclassical.ipynb)) and time-dependent transitions
([§6.24](time-dependent-perturbation.ipynb)) complete the toolkit.

A good approximation knows its own limits. The virtue of doing perturbation theory on a computer is not
that the computer does the algebra — it is that you can always ask the exact answer how well you did, and
watch the honest place where "small correction" stops being small.

## Notebook summary

Perturbation theory, checked against exact — and hydrogen's fine structure. The opening of Movement V.

- **First order** {eq}`eq-pt-first`: $E_n^{(1)}=\langle n|V|n\rangle$, the average of the perturbation
  (`numpy.vdot`).
- **Second order** {eq}`eq-pt-second`: $E_n^{(2)}=\sum_{m\ne n}|\langle m|V|n\rangle|^2/(E_n^0-E_m^0)$ —
  virtual transitions, level repulsion ($E_0^{(2)}<0$), and the energy denominators that decide validity.
- **Checked against exact** {eq}`eq-pt-validity`: `numpy.linalg.eigvalsh` on $H_0+\lambda V$ — agreement
  at small $\lambda$, error $\sim\lambda^{k+1}$, breakdown when levels are close.
- **Degenerate PT** {eq}`eq-pt-degenerate`: `numpy.linalg.eigh` on $V$ within the degenerate subspace —
  the splittings and the good states (the "secular equation," [§6.5](postulates.ipynb)), matching exact to $10^{-3}$.
- **Fine structure** {eq}`eq-fine-structure`: three $\alpha^2$ corrections give $E_{n,j}=-(\text{Ry}/n^2)
  [1+(\alpha^2/n^2)(n/(j+\tfrac12)-\tfrac34)]$ — the $l$-degeneracy of [§6.17](hydrogen-atom.ipynb) lifted, split by $j$ ($2P_{3/2}$
  above $2P_{1/2}$ by $\sim0.045\,$meV), answering [§6.18](spin-magnetic.ipynb)/[§6.19](addition-angular-momenta.ipynb).

We watched the approximation hold and fail against the exact answer, and used it to break hydrogen's
perfect degeneracy. Movement V is under way.

## Outlook

- **The variational method ([§6.22](variational-method.ipynb))**: bounding ground-state energies from above, and the helium atom / a
  glimpse of variational Monte Carlo.
- **The WKB / semiclassical approximation ([§6.23](wkb-semiclassical.ipynb))**: and its connection to the Hamilton–Jacobi theory of
  Volume II ([§2.10](../02-classical-mechanics/hamilton-jacobi.ipynb)).
- **Time-dependent perturbation theory and Fermi's golden rule ([§6.24](time-dependent-perturbation.ipynb))**: transitions, selection rules,
  and spectra.
- **The Lamb shift and QED corrections** beyond fine structure; hyperfine structure (horizons).
- **Cross-reference** [§6.5](postulates.ipynb) (eigenspace projection / the good basis), [§6.17](hydrogen-atom.ipynb) (the $l$-degeneracy), [§6.18](spin-magnetic.ipynb)/[§6.19](addition-angular-momenta.ipynb)
  (spin–orbit, $\mathbf L\cdot\mathbf S$), [§6.6](pauli-uncertainty.ipynb) (compatible observables), and forward to [§6.22](variational-method.ipynb), [§6.23](wkb-semiclassical.ipynb), [§6.24](time-dependent-perturbation.ipynb).

In [ ]:
from ecp.style import footer

footer()